# Data Processing Notebook
This notebook loads the pre-parsed NDJSON (one trial per line) and applies basic cleaning
steps so downstream modeling has consistent types.

## Step 1 · Load the NDJSON dataset
Use pandas to read the pre-parsed trials data so each row corresponds to a single study.


In [ ]:
# Load libraries needed for data IO and wrangling
import json
from pathlib import Path
import pandas as pd

# Read the NDJSON export (one trial per line) into a DataFrame
DATA_PATH = Path('../data/trials_summary.ndjson')
assert DATA_PATH.exists(), f'{DATA_PATH} not found'
df = pd.read_json(DATA_PATH, lines=True)

# Quick peek to confirm the schema loaded as expected
df.head()

,xml_path,nct_id,org_study_id,brief_title,official_title,overall_status,why_stopped,phase,study_type,lead_sponsor,...,conditions,condition_mesh_terms,keywords,interventions,intervention_mesh_terms,primary_outcomes,secondary_outcomes,number_of_arms,number_of_groups,locations
0,/Users/leo/Desktop/520 Project 2/ctg-public-xm...,NCT00000102,NCRR-M01RR01070-0506,Congenital Adrenal Hyperplasia: Calcium Channe...,,Completed,,Phase 1/Phase 2,Interventional,National Center for Research Resources (NCRR),...,[Congenital Adrenal Hyperplasia],"[Adrenal Hyperplasia, Congenital]",[],"[{'type': 'Drug', 'name': 'Nifedipine', 'descr...",[Nifedipine],[],[],,,"{'facility_count': 1, 'countries': ['United St..."
1,/Users/leo/Desktop/520 Project 2/ctg-public-xm...,NCT00000104,NCRR-M01RR00400-0587,Does Lead Burden Alter Neuropsychological Deve...,,Completed,,,Observational,National Center for Research Resources (NCRR),...,[Lead Poisoning],[Lead Poisoning],[lead overburden],"[{'type': 'Procedure', 'name': 'ERP measures o...",[],[],[],,,"{'facility_count': 1, 'countries': ['United St..."
2,/Users/leo/Desktop/520 Project 2/ctg-public-xm...,NCT00000105,2002LS032,Vaccination With Tetanus and KLH to Assess Imm...,Vaccination With Tetanus Toxoid and Keyhole Li...,Terminated,Replaced by another study.,,Observational,"Masonic Cancer Center, University of Minnesota",...,[Cancer],[Neoplasms],[],"[{'type': 'Biological', 'name': 'Intracel KLH ...","[keyhole-limpet hemocyanin, montanide ISA 51, ...",[To assess whether patients can mediate an app...,[Tetanus Response],,3,"{'facility_count': 1, 'countries': ['United St..."
3,/Users/leo/Desktop/520 Project 2/ctg-public-xm...,NCT00000106,NCRR-M01RR03186-9943,41.8 Degree Centigrade Whole Body Hyperthermia...,,Unknown status,,N/A,Interventional,National Center for Research Resources (NCRR),...,[Rheumatic Diseases],[Rheumatic Diseases],[Rheumatoid Diseases],"[{'type': 'Device', 'name': 'Whole body hypert...",[],[],[],,,"{'facility_count': 1, 'countries': ['United St..."
4,/Users/leo/Desktop/520 Project 2/ctg-public-xm...,NCT00000107,NCRR-M01RR00109-0737,Body Water Content in Cyanotic Congenital Hear...,,Completed,,,Observational,National Center for Research Resources (NCRR),...,"[Heart Defects, Congenital]","[Heart Defects, Congenital]",[Cyanotic Congenital Heart Disease],[],[],[],[],,,"{'facility_count': 1, 'countries': ['United St..."


## Step 2 · Normalize date columns
Convert all relevant date strings into pandas `datetime` objects for consistent comparisons later.


In [ ]:
# Convert the most useful date columns into pandas datetime objects
DATE_COLS = [
    "study_first_posted",
    "last_update_posted",
    "start_date",
    "completion_date",
    "primary_completion_date",
]
for col in DATE_COLS:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

df[DATE_COLS].head()


/var/folders/g_/964t0wy17cnbgv5b32sc9sv80000gn/T/ipykernel_47307/2790210628.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")
/var/folders/g_/964t0wy17cnbgv5b32sc9sv80000gn/T/ipykernel_47307/2790210628.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")
/var/folders/g_/964t0wy17cnbgv5b32sc9sv80000gn/T/ipykernel_47307/2790210628.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")


,xml_path,nct_id,org_study_id,brief_title,official_title,overall_status,why_stopped,phase,study_type,lead_sponsor,...,primary_outcomes,secondary_outcomes,number_of_arms,number_of_groups,locations,minimum_age_years,maximum_age_years,enrollment_num,facility_count,conditions_count
0,/Users/leo/Desktop/520 Project 2/ctg-public-xm...,NCT00000102,NCRR-M01RR01070-0506,Congenital Adrenal Hyperplasia: Calcium Channe...,,Completed,,Phase 1/Phase 2,Interventional,National Center for Research Resources (NCRR),...,[],[],,,"{'facility_count': 1, 'countries': ['United St...",14.0,35.0,NaN,1,1
1,/Users/leo/Desktop/520 Project 2/ctg-public-xm...,NCT00000104,NCRR-M01RR00400-0587,Does Lead Burden Alter Neuropsychological Deve...,,Completed,,,Observational,National Center for Research Resources (NCRR),...,[],[],,,"{'facility_count': 1, 'countries': ['United St...",0.0,NaN,NaN,1,1
2,/Users/leo/Desktop/520 Project 2/ctg-public-xm...,NCT00000105,2002LS032,Vaccination With Tetanus and KLH to Assess Imm...,Vaccination With Tetanus Toxoid and Keyhole Li...,Terminated,Replaced by another study.,,Observational,"Masonic Cancer Center, University of Minnesota",...,[To assess whether patients can mediate an app...,[Tetanus Response],,3,"{'facility_count': 1, 'countries': ['United St...",18.0,NaN,112.0,1,1
3,/Users/leo/Desktop/520 Project 2/ctg-public-xm...,NCT00000106,NCRR-M01RR03186-9943,41.8 Degree Centigrade Whole Body Hyperthermia...,,Unknown status,,N/A,Interventional,National Center for Research Resources (NCRR),...,[],[],,,"{'facility_count': 1, 'countries': ['United St...",18.0,65.0,NaN,1,1
4,/Users/leo/Desktop/520 Project 2/ctg-public-xm...,NCT00000107,NCRR-M01RR00109-0737,Body Water Content in Cyanotic Congenital Hear...,,Completed,,,Observational,National Center for Research Resources (NCRR),...,[],[],,,"{'facility_count': 1, 'countries': ['United St...",17.0,60.0,NaN,1,1


## Step 6 · Build labels and feature lists
Create the binary target (`1` for Completed, `0` otherwise) and define which numeric/categorical
columns will feed the models.


In [ ]:
# Binary label: 1 if overall_status == 'Completed', else 0
LABEL_COL = "status_label"
df[LABEL_COL] = (df["overall_status"].fillna("") == "Completed").astype(int)

drop_cols = [LABEL_COL]
feature_cols_num = [
    "minimum_age_years",
    "maximum_age_years",
    "enrollment_num",
    "facility_count",
    "conditions_count",
]
feature_cols_cat = [
    "study_type",
    "phase",
    "gender",
    "healthy_volunteers",
    "lead_sponsor",
]

X_num = df[feature_cols_num]
X_cat = df[feature_cols_cat]
y = df[LABEL_COL]

X_num.head(), X_cat.head(), y.value_counts()


## Step 7 · Train/test split and preprocessing pipeline
Use a ColumnTransformer to impute missing values and one-hot encode categorical fields before feeding models.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# Build the preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, feature_cols_num),
        ("cat", categorical_transformer, feature_cols_cat),
    ]
)

# Split data (stratified to maintain label balance)
X = df[feature_cols_num + feature_cols_cat]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape


## Step 8 · Train multiple classifiers
Fit Logistic Regression, Random Forest, and XGBoost models using the shared preprocessing pipeline.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "logreg": LogisticRegression(max_iter=1000, n_jobs=None),
    "rf": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    "xgb": XGBClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
    ),
}

trained_models = {}
for name, estimator in models.items():
    clf = Pipeline(steps=[("preprocess", preprocessor), ("model", estimator)])
    clf.fit(X_train, y_train)
    trained_models[name] = clf

trained_models


## Step 9 · Evaluate models on the holdout set
Compute accuracy, precision, recall, F1, and ROC-AUC for each classifier. Also build a simple
mean-probability ensemble across all models.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

results = []
probas = []
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    probas.append(y_proba)
    results.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, y_proba),
        }
    )

# Simple average ensemble across all model probabilities
ensemble_proba = np.mean(probas, axis=0)
ensemble_pred = (ensemble_proba >= 0.5).astype(int)
results.append(
    {
        "model": "ensemble_mean",
        "accuracy": accuracy_score(y_test, ensemble_pred),
        "precision": precision_score(y_test, ensemble_pred, zero_division=0),
        "recall": recall_score(y_test, ensemble_pred, zero_division=0),
        "f1": f1_score(y_test, ensemble_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, ensemble_proba),
    }
)

results_df = pd.DataFrame(results).sort_values(by="roc_auc", ascending=False)
results_df


In [29]:
print(df['overall_status'].unique())
print(sum(df['overall_status'] == 'Completed'))
print(df['enrollment_num'].nunique())
print(df['phase'].unique())
print(sum(df['phase'] == 'Phase 3'))

['Completed' 'Terminated' 'Unknown status' 'Withdrawn' 'Recruiting'
 'Active, not recruiting' 'Enrolling by invitation' 'Suspended']
5570
717
['Phase 1/Phase 2' '' 'N/A' 'Phase 1' 'Phase 3' 'Phase 2'
 'Phase 2/Phase 3' 'Phase 4' 'Early Phase 1']
1056


## Step 3 · Engineer numeric-friendly columns
Extract numerical representations for ages, enrollment counts, facility coverage, and condition breadth.


In [25]:
import re
import numpy as np

# Convert textual age expressions ("18 Years", "6 Months") into numeric years
AGE_PATTERN = re.compile(r"(\d+)")


def _age_to_years(value):
    if pd.isna(value) or value in ("", "N/A"):
        return np.nan
    text = str(value)
    match = AGE_PATTERN.search(text)
    if not match:
        return np.nan
    number = float(match.group(1))
    text_lower = text.lower()
    if "month" in text_lower:
        return number / 12
    if "week" in text_lower:
        return number / 52
    if "day" in text_lower:
        return number / 365
    return number


df["minimum_age_years"] = df["minimum_age"].apply(_age_to_years)
df["maximum_age_years"] = df["maximum_age"].apply(_age_to_years)

# Enrollment counts sometimes come as strings; coerce to numeric
df["enrollment_num"] = pd.to_numeric(df["enrollment"], errors="coerce")

# Pull facility counts out of the nested `locations` dict

def _extract_facility_count(value):
    if isinstance(value, dict):
        return value.get("facility_count")
    return np.nan


df["facility_count"] = df["locations"].apply(_extract_facility_count)

# Track how many conditions are listed per trial

df["conditions_count"] = df["conditions"].apply(lambda x: len(x) if isinstance(x, list) else 0)

df[[
    "minimum_age_years",
    "maximum_age_years",
    "enrollment_num",
    "facility_count",
    "conditions_count",
]].head()


,minimum_age_years,maximum_age_years,enrollment_num,facility_count,conditions_count
0,14.0,35.0,NaN,1,1
1,0.0,NaN,NaN,1,1
2,18.0,NaN,112.0,1,1
3,18.0,65.0,NaN,1,1
4,17.0,60.0,NaN,1,1


## Step 4 · Quick sanity checks
Review descriptive statistics for the engineered numeric columns to spot obvious anomalies.


In [26]:
numeric_cols = [
    "minimum_age_years",
    "maximum_age_years",
    "enrollment_num",
    "facility_count",
    "conditions_count",
]

# Display descriptive stats (count, mean, percentiles, etc.)
df[numeric_cols].describe()


,minimum_age_years,maximum_age_years,enrollment_num,facility_count,conditions_count
count,4777.000000,2778.000000,4401.000000,6504.000000,6504.000000
mean,17.084260,68.627027,614.962054,10.732319,2.114545
std,9.898925,33.316248,4568.029061,33.877276,3.046002
min,0.000000,0.002740,0.000000,0.000000,1.000000
25%,16.000000,50.000000,25.000000,1.000000,1.000000
50%,18.000000,70.000000,60.000000,1.000000,1.000000
75%,18.000000,100.000000,224.000000,4.000000,2.000000
max,85.000000,120.000000,121700.000000,539.000000,127.000000


## Step 5 · Persist the cleaned table
Write the processed DataFrame to Parquet so downstream notebooks can consume a compact, typed dataset.


In [27]:
OUTPUT_PATH = Path("../data/trials_summary_clean.parquet")

# Persist cleaned features for future modeling notebooks
df.to_parquet(OUTPUT_PATH, index=False)
OUTPUT_PATH


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.